In [ ]:
import pandas as pd
from pgmpy.estimators import HillClimbSearch, BicScore
from pgmpy.models import BayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination

data = pd.read_parquet("../../data/benchmark/testing/eth")  # Update with your file

print(data.head())

columns = [
    "fee",
    "total_transferred_value",
    "avg_sent_value",
    "avg_received_value",
    "sum_sent_value",
    "sum_received_value",
    "stddev_sent_value",
    "stddev_received_value",
    "avg_time_between_sent_transactions",
    "avg_time_between_received_transactions",
    "unique_out_degree",
    "unique_in_degree",
    "avg_fee_paid",
    "total_fee_paid",
    "min_fee_paid",
    "max_fee_paid",
    "activity_duration_for_sender",
    "activity_duration_for_receiver",
    "label",
]
data = data[columns]

data = data.dropna()

hc = HillClimbSearch(data)
model = hc.estimate(scoring_method=BicScore(data))

print("Learned Structure:")
print(model.edges())

bayesian_network = BayesianNetwork(model.edges())
bayesian_network.fit(data, estimator=MaximumLikelihoodEstimator)

inference = VariableElimination(bayesian_network)

print(bayesian_network.nodes())

   network_name  label  block_number  transaction_index       fee  \
0         False  False      9.811754           0.046512  0.436668   
1         False  False      9.811768           0.988372  1.224422   
2         False  False      9.811769           0.360465  1.224422   
3         False  False      9.811771           0.906977  1.224422   
4         False  False      9.811771           0.965116  1.224422   

   total_transferred_value  total_input_value  sent_value  received_value  \
0                 1.587862           1.590404    1.590404        1.587862   
1                 0.006616           0.008263    0.008263        0.006616   
2                 0.006616           0.008263    0.008263        0.006616   
3                 0.006616           0.008263    0.008263        0.006616   
4                 0.006616           0.008263    0.008263        0.006616   

   block_timestamp  ...  avg_incoming_acceleration_count  \
0       228.459717  ...                              0.0   
1 

  0%|          | 6/1000000 [03:14<8985:50:18, 32.35s/it]  


Learned Structure:
[('unique_in_degree', 'label'), ('label', 'unique_out_degree'), ('label', 'avg_fee_paid'), ('label', 'max_fee_paid'), ('label', 'total_fee_paid'), ('label', 'min_fee_paid')]
['unique_in_degree', 'label', 'unique_out_degree', 'avg_fee_paid', 'max_fee_paid', 'total_fee_paid', 'min_fee_paid']


In [ ]:
query_result = inference.query(variables=["label"], evidence={"max_fee_paid": 1})
print("Query Result:")
print(query_result)

Query Result:
+--------------+--------------+
| label        |   phi(label) |
+==============+==============+
| label(False) |       0.0438 |
+--------------+--------------+
| label(True)  |       0.9562 |
+--------------+--------------+
